# Wind Forecast Evaluation: AIFS vs IFS vs ERA5
## Analysis of Onshore, Coastal, and Offshore Locations around Ireland

This notebook implements the 8-step evaluation plan:
1. Select locations (onshore, coastal, offshore)
2. Extract historical forecasts and observations
3. Ensure consistent alignment (time, height, units)
4. Compute 10m wind speeds from components
5. Define extreme wind events
6. Evaluate deterministic forecast performance
7. Analyze AIFS ensemble behavior
8. Compare performance across environments

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Try importing GRIB handling libraries
try:
    import cfgrib
    print("cfgrib available")
except ImportError:
    print("cfgrib not available - installing...")
    !pip install cfgrib --break-system-packages
    import cfgrib

try:
    import eccodes
    print("eccodes available")
except ImportError:
    print("eccodes not available")

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

## Step 1: Define Locations

Select representative locations across different environments:
- **Onshore**: Inland weather stations
- **Coastal**: Stations near the coast
- **Offshore**: Buoy locations

In [ ]:
# Define locations based on available data
locations = {
    # Onshore locations (Met Éireann stations)
    'Phoenix_Park': {
        'lat': 53.3638,
        'lon': -6.3436,
        'type': 'onshore',
        'station_id': 175,
        'data_file': 'data/Met eireann/hourly-dublin-pheonixpark/hly175.csv'
    },
    'Shannon_Airport': {
        'lat': 52.7019,
        'lon': -8.9248,
        'type': 'onshore',
        'station_id': 518,
        'data_file': 'data/Met eireann/hourly-clare-shannonairport/hly518.csv'
    },
    'Cork_Airport': {
        'lat': 51.8413,
        'lon': -8.4911,
        'type': 'onshore',
        'station_id': 3904,
        'data_file': 'data/Met eireann/hourly-cork-corkairport/hly3904.csv'
    },
    
    # Coastal locations
    'Malin_Head': {
        'lat': 55.3697,
        'lon': -7.3386,
        'type': 'coastal',
        'station_id': 1575,
        'data_file': 'data/Met eireann/hourly-donegal-malinhead/hly1575.csv'
    },
    'Valentia': {
        'lat': 51.9386,
        'lon': -10.2431,
        'type': 'coastal',
        'station_id': 2275,
        'data_file': 'data/Met eireann/hourly-kerry-valentia/hly2275.csv'
    },
    'Mace_Head': {
        'lat': 53.3264,
        'lon': -9.9039,
        'type': 'coastal',
        'station_id': 275,
        'data_file': 'data/Met eireann/houryl-galway-macehead/hly275.csv'
    },
    'Sherkin_Island': {
        'lat': 51.4786,
        'lon': -9.4283,
        'type': 'coastal',
        'station_id': 775,
        'data_file': 'data/Met eireann/hourly-cork-sherkin/hly775.csv'
    },
    
    # Offshore locations (Marine buoys - coordinates to be extracted from buoy data)
    'M2_Buoy': {
        'lat': 53.49,  # Placeholder - extract from buoy_long_lat.png or data
        'lon': -5.43,
        'type': 'offshore',
        'buoy_id': 'M2',
        'data_file': 'data/Bouy data/m2.pdf'  # Will need to extract data
    },
    'M3_Buoy': {
        'lat': 53.23,
        'lon': -11.20,
        'type': 'offshore',
        'buoy_id': 'M3',
        'data_file': 'data/Bouy data/m3.pdf'
    },
    'M4_Buoy': {
        'lat': 51.22,
        'lon': -6.70,
        'type': 'offshore',
        'buoy_id': 'M4',
        'data_file': 'data/Bouy data/m4.pdf'
    },
    'M5_Buoy': {
        'lat': 53.07,
        'lon': -6.70,
        'type': 'offshore',
        'buoy_id': 'M5',
        'data_file': 'data/Bouy data/m5.pdf'
    },
    'M6_Buoy': {
        'lat': 51.69,
        'lon': -10.55,
        'type': 'offshore',
        'buoy_id': 'M6',
        'data_file': 'data/Bouy data/m6.pdf'
    }
}

# Convert to DataFrame for easy visualization
locations_df = pd.DataFrame(locations).T
print(f"\nTotal locations: {len(locations)}")
print(f"Onshore: {sum(locations_df['type'] == 'onshore')}")
print(f"Coastal: {sum(locations_df['type'] == 'coastal')}")
print(f"Offshore: {sum(locations_df['type'] == 'offshore')}")
print("\nLocation Summary:")
print(locations_df[['lat', 'lon', 'type']])

## Step 2 & 3: Load and Align Data

Extract data ensuring:
- UTC time alignment
- 10m height for all wind measurements
- Consistent units (m/s)

In [ ]:
def load_met_eireann_station(filepath):
    """
    Load Met Éireann hourly station data.
    Returns DataFrame with datetime index and wind speed in m/s.
    """
    try:
        # Read the CSV file
        df = pd.read_csv(filepath, skiprows=0)
        
        # Parse datetime - Met Éireann format varies, need to check actual format
        if 'date' in df.columns:
            df['datetime'] = pd.to_datetime(df['date'])
        elif 'Date' in df.columns:
            df['datetime'] = pd.to_datetime(df['Date'])
        
        df.set_index('datetime', inplace=True)
        
        # Extract wind speed (check column names)
        wind_cols = [col for col in df.columns if 'wind' in col.lower() or 'wdsp' in col.lower()]
        
        if wind_cols:
            # Assuming wind speed is in knots or m/s - need to verify
            df['wind_speed'] = df[wind_cols[0]]
            
        return df
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return None

def load_era5_at_location(lat, lon, era5_file='data/era5/era5_initial_state.nc'):
    """
    Load ERA5 data at specific location.
    Returns u10 and v10 components.
    """
    try:
        ds = xr.open_dataset(era5_file)
        
        # Select nearest grid point to location
        era5_point = ds.sel(latitude=lat, longitude=lon, method='nearest')
        
        # Extract u10 and v10
        u10 = era5_point['u10'].values if 'u10' in era5_point else None
        v10 = era5_point['v10'].values if 'v10' in era5_point else None
        
        return u10, v10, era5_point['time'].values
    except Exception as e:
        print(f"Error loading ERA5 at ({lat}, {lon}): {e}")
        return None, None, None

def load_forecast_at_location(lat, lon, forecast_file, model_type='aifs'):
    """
    Load forecast data (AIFS or IFS) at specific location.
    Returns u10 and v10 components at different lead times.
    """
    try:
        # Open GRIB2 file
        ds = xr.open_dataset(forecast_file, engine='cfgrib',
                             backend_kwargs={'filter_by_keys': {'typeOfLevel': 'heightAboveGround'}})
        
        # Select location
        forecast_point = ds.sel(latitude=lat, longitude=lon, method='nearest')
        
        # Extract wind components
        u10 = forecast_point['u10'].values if 'u10' in forecast_point else None
        v10 = forecast_point['v10'].values if 'v10' in forecast_point else None
        times = forecast_point['time'].values if 'time' in forecast_point else None
        steps = forecast_point['step'].values if 'step' in forecast_point else None
        
        return u10, v10, times, steps
    except Exception as e:
        print(f"Error loading {model_type} forecast at ({lat}, {lon}): {e}")
        return None, None, None, None

# Test loading one station
print("Testing data loading...")
test_station = load_met_eireann_station('data/Met eireann/hourly-dublin-pheonixpark/hly175.csv')
if test_station is not None:
    print(f"\nPhoenix Park data loaded: {len(test_station)} records")
    print(f"Date range: {test_station.index.min()} to {test_station.index.max()}")
    print(f"\nColumns: {test_station.columns.tolist()}")
else:
    print("Failed to load test station - check file format")

## Step 4: Compute Wind Speed from Components

In [ ]:
def compute_wind_speed(u, v):
    """
    Compute wind speed from u and v components.
    
    Parameters:
    -----------
    u : array-like
        Eastward wind component (m/s)
    v : array-like
        Northward wind component (m/s)
    
    Returns:
    --------
    wind_speed : array-like
        Wind speed magnitude (m/s)
    """
    return np.sqrt(u**2 + v**2)

def compute_wind_direction(u, v):
    """
    Compute wind direction from u and v components.
    
    Returns direction in degrees (0-360) where:
    0° = North, 90° = East, 180° = South, 270° = West
    """
    direction = np.arctan2(-u, -v) * 180 / np.pi
    direction = (direction + 360) % 360
    return direction

# Test wind speed computation
u_test = np.array([5, -5, 0, 3])
v_test = np.array([0, 0, 5, 4])
ws_test = compute_wind_speed(u_test, v_test)
wd_test = compute_wind_direction(u_test, v_test)

print("Wind Speed Computation Test:")
for i in range(len(u_test)):
    print(f"u={u_test[i]:6.2f}, v={v_test[i]:6.2f} -> speed={ws_test[i]:6.2f} m/s, direction={wd_test[i]:6.1f}°")

## Step 5: Define Extreme Wind Events

In [ ]:
def define_extreme_events(wind_speed_series, method='percentile', threshold=None, percentile=90):
    """
    Define extreme wind events using various criteria.
    
    Parameters:
    -----------
    wind_speed_series : pd.Series or array
        Time series of wind speeds
    method : str
        'percentile': Use percentile threshold
        'absolute': Use absolute threshold (e.g., gale force)
        'both': Use both methods
    threshold : float
        Absolute threshold in m/s (e.g., 17.2 m/s for gale force 8)
    percentile : float
        Percentile for extreme events (e.g., 90, 95, 99)
    
    Returns:
    --------
    extreme_mask : boolean array
        True where extreme events occur
    threshold_value : float
        The threshold value used
    """
    if method == 'percentile':
        threshold_value = np.nanpercentile(wind_speed_series, percentile)
        extreme_mask = wind_speed_series >= threshold_value
    
    elif method == 'absolute':
        if threshold is None:
            # Default: Beaufort Force 8 (gale) = 17.2 m/s
            threshold_value = 17.2
        else:
            threshold_value = threshold
        extreme_mask = wind_speed_series >= threshold_value
    
    elif method == 'both':
        percentile_threshold = np.nanpercentile(wind_speed_series, percentile)
        absolute_threshold = threshold if threshold is not None else 17.2
        extreme_mask = (wind_speed_series >= percentile_threshold) | (wind_speed_series >= absolute_threshold)
        threshold_value = (percentile_threshold, absolute_threshold)
    
    return extreme_mask, threshold_value

# Common wind speed thresholds (Beaufort scale)
WIND_THRESHOLDS = {
    'moderate_breeze': 5.5,     # Beaufort 4
    'fresh_breeze': 8.0,        # Beaufort 5
    'strong_breeze': 10.8,      # Beaufort 6
    'near_gale': 13.9,          # Beaufort 7
    'gale': 17.2,               # Beaufort 8
    'strong_gale': 20.8,        # Beaufort 9
    'storm': 24.5,              # Beaufort 10
    'violent_storm': 28.5,      # Beaufort 11
    'hurricane': 32.7           # Beaufort 12
}

print("Wind Speed Thresholds (Beaufort Scale):")
for name, speed in WIND_THRESHOLDS.items():
    print(f"{name:20s}: {speed:5.1f} m/s")

## Step 6: Deterministic Forecast Evaluation Metrics

In [ ]:
def calculate_forecast_metrics(forecast, observed, extreme_threshold=None):
    """
    Calculate comprehensive forecast verification metrics.
    
    Parameters:
    -----------
    forecast : array-like
        Forecast values
    observed : array-like
        Observed values
    extreme_threshold : float, optional
        Threshold for extreme event metrics
    
    Returns:
    --------
    metrics : dict
        Dictionary containing all computed metrics
    """
    # Remove NaN values
    valid_mask = ~(np.isnan(forecast) | np.isnan(observed))
    forecast = np.array(forecast)[valid_mask]
    observed = np.array(observed)[valid_mask]
    
    if len(forecast) == 0:
        return None
    
    # Basic metrics
    bias = np.mean(forecast - observed)
    mae = np.mean(np.abs(forecast - observed))
    rmse = np.sqrt(np.mean((forecast - observed)**2))
    correlation = np.corrcoef(forecast, observed)[0, 1]
    
    # Relative errors
    mean_observed = np.mean(observed)
    relative_bias = (bias / mean_observed) * 100 if mean_observed != 0 else np.nan
    
    # R-squared
    ss_res = np.sum((observed - forecast)**2)
    ss_tot = np.sum((observed - np.mean(observed))**2)
    r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan
    
    metrics = {
        'n_samples': len(forecast),
        'bias': bias,
        'mae': mae,
        'rmse': rmse,
        'correlation': correlation,
        'r_squared': r_squared,
        'relative_bias_pct': relative_bias,
        'mean_forecast': np.mean(forecast),
        'mean_observed': mean_observed,
        'std_forecast': np.std(forecast),
        'std_observed': np.std(observed)
    }
    
    # Extreme event metrics if threshold provided
    if extreme_threshold is not None:
        forecast_extreme = forecast >= extreme_threshold
        observed_extreme = observed >= extreme_threshold
        
        # Contingency table
        hits = np.sum(forecast_extreme & observed_extreme)
        misses = np.sum(~forecast_extreme & observed_extreme)
        false_alarms = np.sum(forecast_extreme & ~observed_extreme)
        correct_negatives = np.sum(~forecast_extreme & ~observed_extreme)
        
        # Skill scores
        pod = hits / (hits + misses) if (hits + misses) > 0 else np.nan  # Probability of Detection
        far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan  # False Alarm Ratio
        csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan  # Critical Success Index
        
        metrics.update({
            'extreme_threshold': extreme_threshold,
            'n_extreme_observed': np.sum(observed_extreme),
            'n_extreme_forecast': np.sum(forecast_extreme),
            'hits': hits,
            'misses': misses,
            'false_alarms': false_alarms,
            'correct_negatives': correct_negatives,
            'pod': pod,
            'far': far,
            'csi': csi
        })
    
    return metrics

def print_metrics(metrics, model_name="Model"):
    """
    Print metrics in a formatted way.
    """
    print(f"\n{'='*60}")
    print(f"Forecast Metrics: {model_name}")
    print(f"{'='*60}")
    print(f"Sample size: {metrics['n_samples']}")
    print(f"\nBasic Metrics:")
    print(f"  Bias:        {metrics['bias']:8.3f} m/s ({metrics['relative_bias_pct']:6.2f}%)")
    print(f"  MAE:         {metrics['mae']:8.3f} m/s")
    print(f"  RMSE:        {metrics['rmse']:8.3f} m/s")
    print(f"  Correlation: {metrics['correlation']:8.3f}")
    print(f"  R²:          {metrics['r_squared']:8.3f}")
    
    if 'extreme_threshold' in metrics:
        print(f"\nExtreme Event Metrics (threshold = {metrics['extreme_threshold']:.1f} m/s):")
        print(f"  Observed events:  {metrics['n_extreme_observed']}")
        print(f"  Forecast events:  {metrics['n_extreme_forecast']}")
        print(f"  Hits:             {metrics['hits']}")
        print(f"  Misses:           {metrics['misses']}")
        print(f"  False alarms:     {metrics['false_alarms']}")
        print(f"  POD:              {metrics['pod']:8.3f}")
        print(f"  FAR:              {metrics['far']:8.3f}")
        print(f"  CSI:              {metrics['csi']:8.3f}")

# Test metrics calculation
np.random.seed(42)
test_obs = np.random.normal(10, 3, 100)
test_fcst = test_obs + np.random.normal(0, 1, 100)  # Forecast with some error
test_metrics = calculate_forecast_metrics(test_fcst, test_obs, extreme_threshold=13)
print_metrics(test_metrics, "Test Model")

## Step 7: Ensemble Analysis Functions

In [ ]:
def analyze_ensemble(ensemble_members, observed, extreme_threshold=None):
    """
    Analyze ensemble forecast behavior.
    
    Parameters:
    -----------
    ensemble_members : array-like, shape (n_members, n_times)
        Ensemble forecast members
    observed : array-like, shape (n_times,)
        Observed values
    extreme_threshold : float, optional
        Threshold for extreme event analysis
    
    Returns:
    --------
    ensemble_metrics : dict
        Dictionary containing ensemble-specific metrics
    """
    ensemble_members = np.array(ensemble_members)
    observed = np.array(observed)
    
    # Ensemble mean
    ensemble_mean = np.mean(ensemble_members, axis=0)
    
    # Ensemble spread
    ensemble_spread = np.std(ensemble_members, axis=0)
    
    # Ensemble range
    ensemble_min = np.min(ensemble_members, axis=0)
    ensemble_max = np.max(ensemble_members, axis=0)
    
    # CRPS (Continuous Ranked Probability Score) - simplified version
    # For each time, compute mean absolute difference between obs and all members
    crps = np.mean([np.mean(np.abs(member - observed)) for member in ensemble_members])
    
    metrics = {
        'n_members': ensemble_members.shape[0],
        'ensemble_mean': ensemble_mean,
        'ensemble_spread': ensemble_spread,
        'ensemble_min': ensemble_min,
        'ensemble_max': ensemble_max,
        'mean_spread': np.mean(ensemble_spread),
        'crps': crps
    }
    
    # Reliability: Does the ensemble spread match forecast error?
    forecast_error = np.abs(ensemble_mean - observed)
    metrics['spread_error_ratio'] = np.mean(ensemble_spread) / np.mean(forecast_error)
    
    # Extreme event probabilities
    if extreme_threshold is not None:
        # Probability of exceedance for each time
        exceedance_prob = np.mean(ensemble_members >= extreme_threshold, axis=0)
        observed_extreme = observed >= extreme_threshold
        
        metrics['exceedance_probability'] = exceedance_prob
        metrics['observed_extreme'] = observed_extreme
        
        # Brier Score for extreme events
        brier_score = np.mean((exceedance_prob - observed_extreme.astype(float))**2)
        metrics['brier_score'] = brier_score
        
        # Reliability for probabilistic forecasts
        # Bin probabilities and compute observed frequency in each bin
        prob_bins = np.arange(0, 1.1, 0.1)
        reliability_data = []
        for i in range(len(prob_bins)-1):
            mask = (exceedance_prob >= prob_bins[i]) & (exceedance_prob < prob_bins[i+1])
            if np.sum(mask) > 0:
                obs_freq = np.mean(observed_extreme[mask])
                reliability_data.append({
                    'forecast_prob': (prob_bins[i] + prob_bins[i+1]) / 2,
                    'observed_freq': obs_freq,
                    'n_samples': np.sum(mask)
                })
        metrics['reliability_data'] = reliability_data
    
    return metrics

def plot_ensemble_forecast(time, ensemble_members, ensemble_mean, observed, 
                          extreme_threshold=None, title="Ensemble Forecast"):
    """
    Plot ensemble forecast with spread and observations.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot individual members in light gray
    for member in ensemble_members:
        ax.plot(time, member, color='gray', alpha=0.2, linewidth=0.5)
    
    # Plot ensemble mean
    ax.plot(time, ensemble_mean, 'b-', linewidth=2, label='Ensemble Mean')
    
    # Plot ensemble spread (10th-90th percentile)
    p10 = np.percentile(ensemble_members, 10, axis=0)
    p90 = np.percentile(ensemble_members, 90, axis=0)
    ax.fill_between(time, p10, p90, color='blue', alpha=0.2, label='10-90th Percentile')
    
    # Plot observations
    ax.plot(time, observed, 'r-', linewidth=2, label='Observed')
    
    # Plot extreme threshold
    if extreme_threshold is not None:
        ax.axhline(y=extreme_threshold, color='orange', linestyle='--', 
                  linewidth=2, label=f'Extreme Threshold ({extreme_threshold} m/s)')
    
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Wind Speed (m/s)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

print("Ensemble analysis functions loaded.")

## Step 8: Comparative Analysis Across Environments

In [ ]:
def compare_environments(results_by_location, locations_df):
    """
    Compare forecast performance across onshore, coastal, and offshore environments.
    
    Parameters:
    -----------
    results_by_location : dict
        Dictionary with location names as keys and metric dictionaries as values
    locations_df : pd.DataFrame
        DataFrame with location metadata including 'type'
    
    Returns:
    --------
    summary_df : pd.DataFrame
        Summary statistics by environment type
    """
    # Organize results by environment type
    env_results = {'onshore': [], 'coastal': [], 'offshore': []}
    
    for loc_name, metrics in results_by_location.items():
        if loc_name in locations_df.index:
            env_type = locations_df.loc[loc_name, 'type']
            env_results[env_type].append(metrics)
    
    # Compute statistics for each environment
    summary_data = []
    for env_type, metrics_list in env_results.items():
        if len(metrics_list) > 0:
            summary_data.append({
                'environment': env_type,
                'n_locations': len(metrics_list),
                'mean_bias': np.mean([m['bias'] for m in metrics_list]),
                'mean_mae': np.mean([m['mae'] for m in metrics_list]),
                'mean_rmse': np.mean([m['rmse'] for m in metrics_list]),
                'mean_correlation': np.mean([m['correlation'] for m in metrics_list]),
                'std_bias': np.std([m['bias'] for m in metrics_list]),
                'std_rmse': np.std([m['rmse'] for m in metrics_list])
            })
            
            # Add extreme event metrics if available
            if 'pod' in metrics_list[0]:
                summary_data[-1].update({
                    'mean_pod': np.mean([m['pod'] for m in metrics_list if not np.isnan(m['pod'])]),
                    'mean_far': np.mean([m['far'] for m in metrics_list if not np.isnan(m['far'])]),
                    'mean_csi': np.mean([m['csi'] for m in metrics_list if not np.isnan(m['csi'])])
                })
    
    summary_df = pd.DataFrame(summary_data)
    return summary_df

def plot_environment_comparison(summary_df, metric='rmse'):
    """
    Create bar plot comparing metric across environments.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(summary_df))
    metric_col = f'mean_{metric}'
    std_col = f'std_{metric}' if f'std_{metric}' in summary_df.columns else None
    
    bars = ax.bar(x, summary_df[metric_col], 
                  yerr=summary_df[std_col] if std_col else None,
                  capsize=5, alpha=0.7, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
    
    ax.set_xlabel('Environment Type', fontsize=12)
    ax.set_ylabel(f'{metric.upper()} (m/s)', fontsize=12)
    ax.set_title(f'Forecast Error by Environment: {metric.upper()}', 
                fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df['environment'])
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.2f}',
               ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    return fig

print("Comparison functions loaded.")

## Main Analysis Pipeline

Now we'll run the complete analysis for all locations and models.

In [ ]:
# This cell will contain the main analysis loop
# It will be populated once we verify data loading works correctly

print("Main analysis pipeline ready.")
print("\nNext steps:")
print("1. Verify data file formats and adjust loading functions")
print("2. Extract forecast data at each location")
print("3. Compute metrics for each model and location")
print("4. Generate comparison plots and summary tables")